In [30]:
import pandas as pd
data = pd.read_csv("train.csv")
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [34]:
def numerical_imputation(data):
    num_cols = data.select_dtypes(include=["int64", "float64"]).columns
    for col in num_cols:
        data[f"{col}_missing"] = data[col].isnull().astype(int)
        data[col].fillna(data[col].median(), inplace=True)
    return data
def categorical_imputation(data):
    cat_cols = data.select_dtypes(include=["object","category"]).columns
    for col in cat_cols:
        data[col].fillna('Missing', inplace=True)
    return data
def generic_imputation(data,col):
    if data[col].dtype in ['int64', 'float64']:
        data[f"{col}_missing"] = data[col].isnull().astype(int)
        data[col].fillna(data[col].median(), inplace=True)
    elif data[col].dtype in ['object','category']:
        data[col].fillna('Missing', inplace=True)
    elif data[col].dtype == bool:
        data[col].fillna(data[col].mode()[0], inplace=True)
    return data
def null_percentage(data):
    missing = data.isnull().sum()
    percent = missing / data.isnull().count() *100 
    percent = percent[percent > 0].sort_values(ascending=False).round(2)
    print(percent)
    return percent
def clean_option(data,col, threshold, cur_percentage):
    if cur_percentage > threshold:
        data = generic_imputation(data,col)
    else:
        data.drop(index=data[data[col].isnull()].index, inplace=True)    
    return data 
    

In [32]:
import pandas as pd
data = pd.read_csv("train.csv")
data.head()

percentage = null_percentage(data)
print(percentage)


CryoSleep       2.50
ShoppingMall    2.39
VIP             2.34
HomePlanet      2.31
Name            2.30
Cabin           2.29
VRDeck          2.16
FoodCourt       2.11
Spa             2.11
Destination     2.09
RoomService     2.08
Age             2.06
dtype: float64
CryoSleep       2.50
ShoppingMall    2.39
VIP             2.34
HomePlanet      2.31
Name            2.30
Cabin           2.29
VRDeck          2.16
FoodCourt       2.11
Spa             2.11
Destination     2.09
RoomService     2.08
Age             2.06
dtype: float64


In [38]:
for cols in percentage.index:
    data = clean_option(data, cols, 3, percentage[cols])
print(null_percentage(data))
data.head()

Series([], dtype: float64)
Series([], dtype: float64)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [39]:
import numpy as np
import pandas as pd
def data_preprocessing(path: str):
    data = pd.read_csv(path)
    data.head()
    data["PassengerGroup"] = data["PassengerId"].str.split("_").str[0]
    data["PassengerNum"] = pd.to_numeric(data["PassengerId"].str.split("_").str[1], errors="coerce")
    data["Surname"] = data["Name"].str.split(" ").str[-1]
    cabin_split = data["Cabin"].str.split("/", expand=True)
    data["Deck"] = cabin_split[0]
    data["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
    data["Side"] = cabin_split[2]
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    for col in spend_cols:
        data[col] = data[col].fillna(0)
    data["TotalSpend"] = data[spend_cols].sum(axis=1)
    data["Age"] = data["Age"].fillna(data["Age"].median())
    for col in ["CryoSleep", "VIP"]:
        if col in data.columns:
            data[col] = data[col].fillna(data[col].mode()[0])
    cat_cols = ["HomePlanet", "Destination", "Deck", "Side", "Surname", "PassengerGroup"]
    for col in cat_cols:
        data[col] = data[col].fillna("Missing")
    drop_cols = ["Name", "Cabin"]
    if "Transported" in data.columns:
        train_label = data["Transported"].astype(int)
        drop_cols.append("Transported")
    else:
        train_label = None
    train_data = data.drop(columns=drop_cols)
    return train_data, train_label

train_data, train_label = data_preprocessing("train.csv")
train_data.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\2558518375.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,PassengerGroup,PassengerNum,Surname,Deck,CabinNum,Side,TotalSpend
0,0001_01,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,0001,1,Ofracculy,B,0.0,P,0.0
1,0002_01,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,0002,1,Vines,F,0.0,S,736.0
2,0003_01,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,0003,1,Susent,A,0.0,S,10383.0
3,0003_02,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,0003,2,Susent,A,0.0,S,5176.0
4,0004_01,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,0004,1,Santantines,F,1.0,S,1091.0


In [40]:
cat_feature = ["HomePlanet", "CryoSleep", "Destination", "Deck", "Side", "Surname", "VIP", "PassengerGroup"]

n_splits = 5
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
if train_label is not None:
    fold_accuracies = []
    train_features = train_data.drop(columns=["PassengerId"])
    for fold, (train_index, val_index) in enumerate(skf.split(train_features, train_label)):
        X_train, X_val = train_features.iloc[train_index], train_features.iloc[val_index]
        y_train, y_val = train_label.iloc[train_index], train_label.iloc[val_index]
        cat = CatBoostClassifier(
            iterations=1000,
            depth=8,
            learning_rate=0.03,
            loss_function="Logloss",
            eval_metric="Accuracy",
            random_seed=42,
            verbose=100,
            task_type="GPU"
        )
        cat.fit(X_train, y_train, cat_features=cat_feature, eval_set=(X_val, y_val), verbose=50)
        preds = cat.predict(X_val)
        acc = accuracy_score(y_val, preds)
        fold_accuracies.append(acc)
        print(f"Fold {fold + 1} Accuracy: {acc:.4f}")
    print(f"Average Accuracy across {n_splits} folds: {np.mean(fold_accuracies):.4f}")

0:	learn: 0.7745183	test: 0.7682576	best: 0.7682576 (0)	total: 125ms	remaining: 2m 5s
50:	learn: 0.8078804	test: 0.8039103	best: 0.8044853 (48)	total: 5.38s	remaining: 1m 40s
100:	learn: 0.8185217	test: 0.8113859	best: 0.8119609 (93)	total: 10.7s	remaining: 1m 34s
150:	learn: 0.8247052	test: 0.8142611	best: 0.8159862 (148)	total: 15.9s	remaining: 1m 29s
200:	learn: 0.8294507	test: 0.8159862	best: 0.8165612 (186)	total: 21s	remaining: 1m 23s
250:	learn: 0.8337647	test: 0.8148361	best: 0.8165612 (186)	total: 26.1s	remaining: 1m 17s
300:	learn: 0.8395168	test: 0.8177113	best: 0.8188614 (279)	total: 31.2s	remaining: 1m 12s
350:	learn: 0.8425367	test: 0.8194365	best: 0.8200115 (341)	total: 36.2s	remaining: 1m 6s
400:	learn: 0.8445499	test: 0.8228867	best: 0.8234618 (395)	total: 41s	remaining: 1m 1s
450:	learn: 0.8465631	test: 0.8257619	best: 0.8257619 (426)	total: 45.6s	remaining: 55.5s
500:	learn: 0.8497268	test: 0.8274871	best: 0.8274871 (497)	total: 50.5s	remaining: 50.3s
550:	learn: 0.8

In [45]:
test_data, _ = data_preprocessing("test.csv")
passenger_id = test_data["PassengerId"].copy()
test_features = test_data.drop(columns=["PassengerId"])
predictions = cat.predict(test_features)
submission = pd.DataFrame({
    "PassengerId": passenger_id,
    "Transported": predictions.astype(bool)
})
submission.to_csv("submission1.csv", index=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\2558518375.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


In [42]:
#boosting
cat_feature = ["HomePlanet", "CryoSleep", "Destination", "Deck", "Side", "Surname", "VIP", "PassengerGroup"]
train_data, train_label = data_preprocessing("train.csv")
train_features = train_data.drop(columns=["PassengerId"])
X_encoded = train_features.copy()

X_encoded[cat_feature] = X_encoded[cat_feature].astype(str)
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
encoder = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=-1)
X_encoded[cat_feature] = encoder.fit_transform(X_encoded[cat_feature])
cat_indices = [X_encoded.columns.get_loc(col) for col in cat_feature]
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_encoded, train_label, test_size=0.2, random_state=42, stratify=train_label)

model_cat = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=100,
    task_type="GPU",
    early_stopping_rounds=50
)
model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=100)
model_lgbm = LGBMClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    random_state=42,
    verbose=100,
    device="gpu",
    metric="binary_logloss",
)
import lightgbm as lgb
lgbm_early_stopping = lgb.early_stopping(stopping_rounds=50, verbose=False)
model_lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgbm_early_stopping])

model_xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    random_state=42,
    verbosity=1,
    eval_metric="logloss",
    early_stopping_rounds=50
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

probs_cat = model_cat.predict_proba(X_val)[:, 1]
probs_lgbm = model_lgbm.predict_proba(X_val)[:, 1]  # type: ignore
probs_xgb = model_xgb.predict_proba(X_val)[:, 1]

final_probs = (probs_cat + probs_lgbm + probs_xgb) / 3
final_predictions = (final_probs >= 0.5).astype(int)
from sklearn.metrics import accuracy_score
ensemble_accuracy = accuracy_score(y_val, final_predictions)
print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\2558518375.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])


0:	learn: 0.7748059	test: 0.7625072	best: 0.7625072 (0)	total: 87.4ms	remaining: 1m 27s
bestTest = 0.7958596895
bestIteration = 20
Shrink model to first 21 iterations.
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.718579
[LightGBM] [Info] Total Bins 2406
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 17
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 10 dense feature groups (0.08 MB) transferred to GPU in 0.000806 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] 

In [44]:
test_data, _ = data_preprocessing("test.csv")
passenger_id = test_data["PassengerId"].copy()
X_encoded = test_data.drop(columns=["PassengerId"]).copy()
X_encoded[cat_feature] = X_encoded[cat_feature].astype(str)
X_encoded[cat_feature] = encoder.transform(X_encoded[cat_feature])
probs_cat = model_cat.predict_proba(X_encoded)[:, 1]
probs_lgbm = model_lgbm.predict_proba(X_encoded)[:, 1]  # type: ignore
probs_xgb = model_xgb.predict_proba(X_encoded)[:, 1]
final_probs = (probs_cat + probs_lgbm + probs_xgb) / 3
final_predictions = (final_probs >= 0.5).astype(int)
submission = pd.DataFrame({
    "PassengerId": passenger_id,
    "Transported": final_predictions.astype(bool)
})
submission.to_csv("submission2.csv", index=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25644\2558518375.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].fillna(data[col].mode()[0])
